In [ ]:
import duckdb
con = duckdb.connect(database=':memory:')
query = """
    CREATE OR REPLACE VIEW tlc_silver AS 
    SELECT * FROM read_parquet('data/silver/taxi_trips/*/*.parquet', hive_partitioning=true);
"""
con.execute(query)
print(con.execute("SELECT COUNT(*) FROM tlc_silver").fetchone())

(3066766,)


In [ ]:
sql_daily_kpi = """
    SELECT 
        pickup_date,
        COUNT(*) AS total_trips,
        ROUND(SUM(fare_amount), 2) AS total_revenue
    FROM tlc_silver
    GROUP BY pickup_date
    ORDER BY pickup_date ASC;
"""
df_daily_kpi = con.execute(sql_daily_kpi).df()
print("✅ Step 2 运行成功！每日 KPI 数据如下：")
print(df_daily_kpi.head())

✅ Step 2 运行成功！每日 KPI 数据如下：
  pickup_date  total_trips  total_revenue
0  2008-12-31            2          70.00
1  2022-10-24            4         295.00
2  2022-10-25            7         364.00
3  2022-12-31           25         415.70
4  2023-01-01        76752     1678504.53


In [ ]:
sql_hourly_kpi = """
    SELECT 
        DATE_PART('hour', tpep_pickup_datetime) AS pickup_hour,
        COUNT(*) AS trip_volume
    FROM tlc_silver
    GROUP BY DATE_PART('hour', tpep_pickup_datetime)
    ORDER BY pickup_hour ASC;
"""
df_hourly_kpi = con.execute(sql_hourly_kpi).df()
print("✅ Step 3 运行成功！全天最忙碌的 3 个小时是：")
print(df_hourly_kpi.sort_values(by='trip_volume', ascending=False).head(3))

✅ Step 3 运行成功！全天最忙碌的 3 个小时是：
    pickup_hour  trip_volume
2             2       215889
1             1       209493
23           23       196424


In [ ]:

sql_hourly_kpi = """
    SELECT 
        DATE_PART('hour', tpep_pickup_datetime) AS pickup_hour,
        COUNT(*) AS trip_volume
    FROM tlc_silver
    GROUP BY DATE_PART('hour', tpep_pickup_datetime)
    ORDER BY pickup_hour ASC;
"""
df_hourly_kpi = con.execute(sql_hourly_kpi).df()

print("✅ Step 3 运行成功！全天最忙碌的 3 个小时是：")
# 按照订单量从高到低排序，只看前 3 名 [cite: 204, 205]
print(df_hourly_kpi.sort_values(by='trip_volume', ascending=False).head(3))

✅ Step 3 运行成功！全天最忙碌的 3 个小时是：
    pickup_hour  trip_volume
2             2       215889
1             1       209493
23           23       196424


In [ ]:

sql_challenge = """
    WITH daily_counts AS (
        SELECT 
            pickup_date,
            COUNT(*) AS total_trips
        FROM tlc_silver
        GROUP BY pickup_date
    )
    SELECT 
        pickup_date,
        total_trips,
        ROUND(AVG(total_trips) OVER (
            ORDER BY pickup_date 
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ), 2) AS rolling_7d_avg
    FROM daily_counts
    ORDER BY pickup_date ASC;
"""
df_challenge = con.execute(sql_challenge).df()
print("🏆 挑战任务通关！包含7天滚动平均的数据如下：")
print(df_challenge.head(10))

🏆 挑战任务通关！包含7天滚动平均的数据如下：
  pickup_date  total_trips  rolling_7d_avg
0  2008-12-31            2            2.00
1  2022-10-24            4            3.00
2  2022-10-25            7            4.33
3  2022-12-31           25            9.50
4  2023-01-01        76752        15358.00
5  2023-01-02        65777        23761.17
6  2023-01-03        85783        32621.43
7  2023-01-04        95092        46205.71
8  2023-01-05       101063        60642.71
9  2023-01-06       102550        75291.71
